# Setting up kaggle and downloading the data

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
! mkdir ~/.kaggle

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [26]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [27]:
!kaggle competitions download walmart-recruiting-store-sales-forecasting

walmart-recruiting-store-sales-forecasting.zip: Skipping, found more recently modified local copy (use --force to force download)


In [28]:
! unzip /content/walmart-recruiting-store-sales-forecasting.zip


Archive:  /content/walmart-recruiting-store-sales-forecasting.zip
replace features.csv.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           


In [29]:
! unzip /content/walmart-recruiting-store-sales-forecasting.zip
! unzip /content/train.csv.zip
! unzip /content/test.csv.zip
! unzip /content/features.csv.zip
! unzip /content/stores.csv.zip

Archive:  /content/walmart-recruiting-store-sales-forecasting.zip
replace features.csv.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           
Archive:  /content/train.csv.zip
replace train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: train.csv               
Archive:  /content/test.csv.zip
replace test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: test.csv                
Archive:  /content/features.csv.zip
replace features.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: features.csv            
unzip:  cannot find or open /content/stores.csv.zip, /content/stores.csv.zip.zip or /content/stores.csv.zip.ZIP.


In [30]:
# !pip install neuralforecast
# !pip install torch torchvision
# !pip install "sympy<1.6.0"
!pip install neuralforecast torch torchvision sympy --upgrade

# Import libraries

In [31]:
import torch
from torch import nn
from torch.optim import Adam
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import wandb
import torch.nn as nn
import torch.nn.functional as F
from torchsummary import summary
from torchvision import transforms
import torchvision.models as models
import seaborn as sns

import wandb
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


import os
import numpy as np
import pandas as pd
import wandb
from neuralforecast import NeuralForecast
from neuralforecast.models import DLinear
from neuralforecast.losses.pytorch import MAE
import itertools






In [32]:
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU
print("Device available: ", device)

Device available:  cuda


# Load the data

In [33]:
train = pd.read_csv('train.csv')

# preprocessing

In [34]:
def prepare_data(filepath):
    """
    Loads Walmart data and formats it for NeuralForecast.
    NeuralForecast expects: unique_id, ds, y
    """
    df = pd.read_csv(filepath)

    df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
    df['ds'] = pd.to_datetime(df['Date'])
    df['y'] = df['Weekly_Sales']

    return df[['unique_id', 'ds', 'y', 'IsHoliday']]



In [35]:
def split_data(df, validation_weeks=31):
    """
    Splits data based on the last N weeks.
    """
    max_date = df['ds'].max()
    split_date = max_date - pd.Timedelta(weeks=validation_weeks)

    train_df = df[df['ds'] <= split_date].copy()
    val_df = df[df['ds'] > split_date].copy()

    return train_df, val_df


# evaluation

In [36]:
def evaluate_predictions(val_df, preds_df, model_name='DLinear'):
    """
    Merges predictions with actual values and calculates metrics.
    Handles missing target values by dropping them (loss = 0 contribution).
    """

    eval_df = pd.merge(val_df, preds_df, on=['unique_id', 'ds'], how='inner')

    eval_df = eval_df.dropna(subset=['y'])

    if eval_df.empty:
        return {"val_mae": np.nan, "val_wmae": np.nan}

    mae = np.mean(np.abs(eval_df['y'] - eval_df[model_name]))

    bias = np.mean(eval_df['y'] - eval_df[model_name])

    weights = eval_df['IsHoliday'].apply(lambda x: 5 if x else 1)
    wmae = np.sum(weights * np.abs(eval_df['y'] - eval_df[model_name])) / np.sum(weights)

    return {
        "val_mae": mae,
        "val_wmae": wmae,
        "val_bias": bias
    }



# base model

In [14]:
def run_experiment(config=None):
    with wandb.init(project="walmart-sales-forecasting", config=config) as run:
        config = wandb.config

        print("Loading and preparing data...")
        df = prepare_data(config.data_path)
        train_df, val_df = split_data(df, validation_weeks=config.validation_weeks)

        model = DLinear(
            h=config.horizon,
            input_size=config.input_size,
            loss=MAE(),
            max_steps=config.max_steps,
            learning_rate=config.learning_rate,
            random_seed=config.seed,

            accelerator='auto',
            enable_progress_bar=True,
            start_padding_enabled=True
        )

        nf = NeuralForecast(models=[model], freq='W-FRI')

        print("Training DLinear...")
        nf.fit(df=train_df)

        print("Generating forecasts...")
        preds_df = nf.predict()

        print("Evaluating predictions...")
        metrics = evaluate_predictions(val_df, preds_df, model_name='DLinear')

        wandb.log(metrics)
        print(f"Logged Metrics -> MAE: {metrics['val_mae']:.2f} | WMAE: {metrics['val_wmae']:.2f}")



In [15]:
if __name__ == "__main__":
    config_params = {
        "data_path": "train.csv",
        "validation_weeks": 31,
        "horizon": 31,
        "input_size": 52,
        "max_steps": 500,
        "learning_rate": 1e-3,
        "seed": 42
    }

    run_experiment(config=config_params)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: grtsir22 (grtsir22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Generating forecasts...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Evaluating predictions...
Logged Metrics -> MAE: 2094.54 | WMAE: 2116.21


val_bias,▁
val_mae,▁
val_wmae,▁
val_bias,331.9088
val_mae,2094.5394
val_wmae,2116.21339


# longer input size

In [16]:
if __name__ == "__main__":
    config_params = {
        "data_path": "train.csv",
        "validation_weeks": 31,
        "horizon": 31,
        "input_size": 104,
        "max_steps": 500,
        "learning_rate": 1e-3,
        "seed": 42
    }

    run_experiment(config=config_params)

Loading and preparing data...


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | decomp              | SeriesDecomp  | 0      | train
7 | linear_trend        | Linear        | 3.3 K  | train
8 | linear_season       | Linear        | 3.3 K  | train
------------------

Training DLinear...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Generating forecasts...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Evaluating predictions...
Logged Metrics -> MAE: 3415.78 | WMAE: 3765.06


val_bias,▁
val_mae,▁
val_wmae,▁
val_bias,1785.71281
val_mae,3415.78327
val_wmae,3765.06341


# grid search for inp size

In [37]:
def train_and_evaluate_local(config, train_df, val_df):
    """Trains a configuration completely locally without calling W&B."""
    model = DLinear(
        h=config["horizon"],
        input_size=config["input_size"],
        moving_avg_window=config["moving_avg_window"],
        loss=MAE(),
        max_steps=config["max_steps"],
        learning_rate=config["learning_rate"],
        random_seed=config["seed"],

        # optimizer=torch.optim.Adagrad,
        # scaler_type='robust',

        accelerator='auto',
        enable_progress_bar=True,
        start_padding_enabled=True,
    )

    nf = NeuralForecast(models=[model], freq='W-FRI')
    nf.fit(df=train_df)
    preds_df = nf.predict()
    metrics = evaluate_predictions(val_df, preds_df, model_name='DLinear')

    return metrics, nf


In [38]:
if __name__ == "__main__":

    base_config = {
        "data_path": "train.csv",
        "validation_weeks": 31,
        "horizon": 31,
        "max_steps": 500,
        "seed": 42
    }

    param_grid = {
        "input_size": [40, 52, 66, 70, 104],
        "moving_avg_window": [ 25],
        "learning_rate": [1e-3]
    }

    keys, values = zip(*param_grid.items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

    print(f"Loading and preparing data...")
    df = prepare_data(base_config["data_path"])
    train_df, val_df = split_data(df, validation_weeks=base_config["validation_weeks"])

    best_wmae = float('inf')
    best_config = None
    best_metrics = None
    best_model_dir = "./models/best_dlinear_walmart"

    print(f"Starting Local Grid Search across {len(experiments)} combinations...")

    for i, exp in enumerate(experiments):
        current_config = {**base_config, **exp}
        print(f"\n--- Running Trial {i+1}/{len(experiments)} ---")
        print(f"Parameters: {exp}")

        metrics, nf = train_and_evaluate_local(current_config, train_df, val_df)
        print(f"Trial Results -> MAE: {metrics['val_mae']:.2f} | WMAE: {metrics['val_wmae']:.2f}")

        if metrics['val_wmae'] < best_wmae:
            best_wmae = metrics['val_wmae']
            best_config = current_config
            best_metrics = metrics

            os.makedirs(best_model_dir, exist_ok=True)
            nf.save(path=best_model_dir, overwrite=True)
            print("New best configuration saved locally!")


Loading and preparing data...


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Starting Local Grid Search across 5 combinations...

--- Running Trial 1/5 ---
Parameters: {'input_size': 40, 'moving_avg_window': 25, 'learning_rate': 0.001}


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | decomp              | SeriesDecomp  | 0      | train
7 | linear_trend        | Linear        | 1.3 K  | train
8 | linear_season       | Linear        | 1.3 K  | train
--------------------------------------------------------------
2.5 K     Trainable params
0         Non-trainable params
2.5 K     Total params
0.010     Total estimated model params size (MB)
11        Modules in train mode
0         M

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Trial Results -> MAE: 3415.78 | WMAE: 3765.06


In [39]:
print("\n==========================================")
print("Grid Search Complete! Logging the CHAMPION model to W&B...")
print(f"Best Config: {best_config}")
print(f"Best WMAE: {best_wmae:.2f}")

with wandb.init(project="walmart-sales-forecasting", config=best_config) as run:

    wandb.log(best_metrics)


Grid Search Complete! Logging the CHAMPION model to W&B...
Best Config: {'data_path': 'train.csv', 'validation_weeks': 31, 'horizon': 31, 'max_steps': 500, 'seed': 42, 'input_size': 52, 'moving_avg_window': 25, 'learning_rate': 0.001}
Best WMAE: 2116.21


val_bias,▁
val_mae,▁
val_wmae,▁
val_bias,331.9088
val_mae,2094.5394
val_wmae,2116.21339


# grid search for window

In [40]:
if __name__ == "__main__":

    base_config = {
        "data_path": "train.csv",
        "validation_weeks": 31,
        "horizon": 31,
        "max_steps": 500,
        "seed": 42
    }

    param_grid = {
        "input_size": [52],
        "moving_avg_window": [ 11,13,15,21,25,29],
        "learning_rate": [1e-3]
    }

    keys, values = zip(*param_grid.items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

    print(f"Loading and preparing data...")
    df = prepare_data(base_config["data_path"])
    train_df, val_df = split_data(df, validation_weeks=base_config["validation_weeks"])

    best_wmae = float('inf')
    best_config = None
    best_metrics = None
    best_model_dir = "./models/best_dlinear_walmart"

    print(f"Starting Local Grid Search across {len(experiments)} combinations...")

    for i, exp in enumerate(experiments):
        current_config = {**base_config, **exp}
        print(f"\n--- Running Trial {i+1}/{len(experiments)} ---")
        print(f"Parameters: {exp}")

        metrics, nf = train_and_evaluate_local(current_config, train_df, val_df)
        print(f"Trial Results -> MAE: {metrics['val_mae']:.2f} | WMAE: {metrics['val_wmae']:.2f}")

        if metrics['val_wmae'] < best_wmae:
            best_wmae = metrics['val_wmae']
            best_config = current_config
            best_metrics = metrics

            os.makedirs(best_model_dir, exist_ok=True)
            nf.save(path=best_model_dir, overwrite=True)
            print("New best configuration saved locally!")


Loading and preparing data...


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | decomp              | SeriesDecomp  | 0      | train
7 | linear_trend        | Linear        | 1.6 K  | train
8 | linear_season       | Linear        | 1.6 K  | train
------------------

Starting Local Grid Search across 6 combinations...

--- Running Trial 1/6 ---
Parameters: {'input_size': 52, 'moving_avg_window': 11, 'learning_rate': 0.001}


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Trial Results -> MAE: 2099.95 | WMAE: 2122.75


In [41]:
print("\n==========================================")
print("Grid Search Complete! Logging the CHAMPION model to W&B...")
print(f"Best Config: {best_config}")
print(f"Best WMAE: {best_wmae:.2f}")

with wandb.init(project="walmart-sales-forecasting", config=best_config) as run:
    wandb.log(best_metrics)


Grid Search Complete! Logging the CHAMPION model to W&B...
Best Config: {'data_path': 'train.csv', 'validation_weeks': 31, 'horizon': 31, 'max_steps': 500, 'seed': 42, 'input_size': 52, 'moving_avg_window': 25, 'learning_rate': 0.001}
Best WMAE: 2116.21


val_bias,▁
val_mae,▁
val_wmae,▁
val_bias,331.9088
val_mae,2094.5394
val_wmae,2116.21339


# grid search steps and learning rate

In [42]:
if __name__ == "__main__":
    base_config = {
        "data_path": "train.csv",
        "validation_weeks": 31,
        "horizon": 31,
        "seed": 42,
        'batch_size' : 512

    }

    param_grid = {
        "max_steps": [500,1000,2600],
        "input_size": [52],
        "moving_avg_window": [ 25],
        "learning_rate": [ 1e-2,1e-3,2e-5]
    }

    keys, values = zip(*param_grid.items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

    print(f"Loading and preparing data...")
    df = prepare_data(base_config["data_path"])
    train_df, val_df = split_data(df)

    best_wmae = float('inf')
    best_config = None
    best_metrics = None
    best_model_dir = "./models/best_dlinear_walmart"

    print(f"Starting Local Grid Search across {len(experiments)} combinations...")

    for i, exp in enumerate(experiments):
        current_config = {**base_config, **exp}
        print(f"\n--- Running Trial {i+1}/{len(experiments)} ---")
        print(f"Parameters: {exp}")

        metrics, nf = train_and_evaluate_local(current_config, train_df, val_df)
        print(f"Trial Results -> MAE: {metrics['val_mae']:.2f} | WMAE: {metrics['val_wmae']:.2f}")

        if metrics['val_wmae'] < best_wmae:
            best_wmae = metrics['val_wmae']
            best_config = current_config
            best_metrics = metrics

            os.makedirs(best_model_dir, exist_ok=True)
            nf.save(path=best_model_dir, overwrite=True)
            print("New best configuration saved locally!")


Loading and preparing data...


INFO:lightning_fabric.utilities.seed:Seed set to 42


Starting Local Grid Search across 9 combinations...

--- Running Trial 1/9 ---
Parameters: {'max_steps': 500, 'input_size': 52, 'moving_avg_window': 25, 'learning_rate': 0.01}


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | decomp              | SeriesDecomp  | 0      | train
7 | linear_trend        | Linear        | 1.6 K  | train
8 | linear_season       | Linear        | 1.6 K  | train
--------------------------------------------------------------
3.3 K  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=2600` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Trial Results -> MAE: 2669.22 | WMAE: 2728.82


In [44]:
print("\n==========================================")
print("Grid Search Complete! Logging the CHAMPION model to W&B...")
print(f"Best Config: {best_config}")
print(f"Best WMAE: {best_wmae:.2f}")

with wandb.init(project="walmart-sales-forecasting", config=best_config) as run:
    wandb.log(best_metrics)

    artifact = wandb.Artifact(
        name="best-dlinear-model",
        type="model",
        description=f"Best DLinear model from local tuning loop. WMAE: {best_wmae:.2f}"
    )
    artifact.add_dir(best_model_dir)
    run.log_artifact(artifact)
    print("Successfully logged the champion metrics and artifact to W&B.")


Grid Search Complete! Logging the CHAMPION model to W&B...
Best Config: {'data_path': 'train.csv', 'validation_weeks': 31, 'horizon': 31, 'seed': 42, 'batch_size': 512, 'max_steps': 500, 'input_size': 52, 'moving_avg_window': 25, 'learning_rate': 0.001}
Best WMAE: 2116.21


wandb: Adding directory to artifact (models/best_dlinear_walmart)... Done. 0.1s


Successfully logged the champion metrics and artifact to W&B.


val_bias,▁
val_mae,▁
val_wmae,▁
val_bias,331.9088
val_mae,2094.5394
val_wmae,2116.21339


# final

In [45]:
def train_local(config, train_df):
    """Trains a configuration completely locally without calling W&B."""
    model = DLinear(
        h=config["horizon"],
        input_size=config["input_size"],
        moving_avg_window=config["moving_avg_window"],
        loss=MAE(),
        max_steps=config["max_steps"],
        learning_rate=config["learning_rate"],
        random_seed=config["seed"],

        # optimizer=torch.optim.Adagrad,
        # scaler_type='robust',

        accelerator='auto',
        enable_progress_bar=True,
        start_padding_enabled=True,
    )

    nf = NeuralForecast(models=[model], freq='W-FRI')
    nf.fit(df=train_df)
    preds_df = nf.predict()

    return nf


In [46]:
if __name__ == "__main__":
    base_config = {
        "data_path": "train.csv",
        "validation_weeks": 40,
        "horizon": 40,
        "seed": 42,
        'batch_size' : 512

    }

    param_grid = {
        "max_steps": [500],
        "input_size": [52],
        "moving_avg_window": [ 25],
        "learning_rate": [ 1e-3]
    }

    keys, values = zip(*param_grid.items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

    print(f"Loading and preparing data...")
    df = prepare_data(base_config["data_path"])

    best_wmae = float('inf')
    best_config = None
    best_metrics = None
    best_model_dir = "./models/best_dlinear_walmart"

    print(f"Starting Local Grid Search across {len(experiments)} combinations...")

    for i, exp in enumerate(experiments):
        current_config = {**base_config, **exp}
        print(f"\n--- Running Trial {i+1}/{len(experiments)} ---")
        print(f"Parameters: {exp}")

        nf = train_local(current_config, df)

        best_config = current_config

        os.makedirs(best_model_dir, exist_ok=True)
        nf.save(path=best_model_dir, overwrite=True)
        print("New best configuration saved locally!")


Loading and preparing data...


INFO:lightning_fabric.utilities.seed:Seed set to 42


Starting Local Grid Search across 1 combinations...

--- Running Trial 1/1 ---
Parameters: {'max_steps': 500, 'input_size': 52, 'moving_avg_window': 25, 'learning_rate': 0.001}


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | decomp              | SeriesDecomp  | 0      | train
7 | linear_trend        | Linear        | 2.1 K  | train
8 | linear_season       | Linear        | 2.1 K  | train
--------------------------------------------------------------
4.2 K  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

New best configuration saved locally!


In [47]:
print("\n==========================================")
print("Grid Search Complete! Logging the CHAMPION model to W&B...")
print(f"Best Config: {best_config}")

with wandb.init(project="walmart-sales-forecasting", config=best_config) as run:

    artifact = wandb.Artifact(
        name="best-DLinear-model-final1",
        type="model",
        description=f"Best DLinear model from local tuning loop. WMAE: {best_wmae:.2f}"
    )
    artifact.add_dir(best_model_dir)
    run.log_artifact(artifact)
    print("Successfully logged the champion metrics and artifact to W&B.")



Grid Search Complete! Logging the CHAMPION model to W&B...
Best Config: {'data_path': 'train.csv', 'validation_weeks': 40, 'horizon': 40, 'seed': 42, 'batch_size': 512, 'max_steps': 500, 'input_size': 52, 'moving_avg_window': 25, 'learning_rate': 0.001}


wandb: Adding directory to artifact (models/best_dlinear_walmart)... Done. 0.0s


Successfully logged the champion metrics and artifact to W&B.
